# 02 · Pipeline end-to-end

**Landing → Bronze → Silver → Gold → Serving** en una sola corrida, sin pasos manuales (consigna §6.1, plan 2.8).

El notebook **no define transformaciones**: importa de `src/` y llama a las mismas funciones que
`python -m src.pipeline` (D17). Correrlo de punta a punta con *Run All*.

| Entorno | Cómo abrirlo |
|---|---|
| Docker (recomendado) | `make up` y abrir <http://localhost:8888> → `notebooks/02_pipeline.ipynb` |
| Línea de comandos, sin notebook | `make pipeline` (o `python -m src.pipeline`) |
| Google Colab + AstraDB (fase 2) | Abrirlo en Colab y correr primero la celda de **Setup de Colab** |

Las salidas se borran antes de commitear: pueden mostrar rutas locales.

## Setup de Colab (fase 2)

Solo hace algo dentro de Google Colab; en Docker o en un venv se saltea sola. Hace falta, **antes de correr**:

1. **Secret `ASTRA_DB_TOKEN`** en Colab (ícono de llave, activar *Notebook access*) con el token `AstraCS:...`. Si no existe, la celda lo pide por prompt sin mostrarlo.
2. **Secure connect bundle** de la base: la celda abre un selector para subir el `.zip`. Queda en `/content`, fuera del repo.

La celda instala Java 17 y las dependencias de `requirements.txt` necesarias para el pipeline, clona el repo en `/content/big-data`, y deja `PIPELINE_PROFILE=cloud` (D17). El lake queda en el disco de la sesión (`data/datalake/` del clon) y se pierde al reciclarse. Ni el token ni el bundle se guardan en el notebook ni en el repo.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/MatiasColeur02/big-data.git"
REPO_BRANCH = "main"
CLONE_DIR = Path("/content/big-data")
BUNDLE = Path("/content/secure-connect-cloud-analytics.zip")
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
PIPELINE_PACKAGES = ("pyspark", "cassandra-driver", "pyyaml")   # del requirements.txt; el resto ya viene en Colab


def sh(command):
    subprocess.run(command, shell=True, check=True)


def java_major():
    out = subprocess.run("java -version", shell=True, capture_output=True, text=True).stderr
    try:
        return int(out.split('"')[1].split(".")[0])
    except (IndexError, ValueError):
        return None


if not IN_COLAB:
    print("Fuera de Colab: no hay nada que preparar.")
else:
    os.chdir("/content")

    # 1. Secure connect bundle (a /content, fuera del repo).
    if not BUNDLE.is_file():
        from google.colab import files

        print("Subir el secure connect bundle (.zip) de AstraDB:")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No se subió el bundle: sin él no hay conexión con AstraDB")
        Path(next(iter(uploaded))).rename(BUNDLE)
    os.environ["ASTRA_BUNDLE_PATH"] = str(BUNDLE)

    # 2. Token: Secret de Colab o prompt oculto. No se imprime ni se guarda.
    if not os.environ.get("ASTRA_DB_TOKEN"):
        try:
            from google.colab import userdata

            os.environ["ASTRA_DB_TOKEN"] = userdata.get("ASTRA_DB_TOKEN")
        except Exception:
            from getpass import getpass

            os.environ["ASTRA_DB_TOKEN"] = getpass("ASTRA_DB_TOKEN (AstraCS:...): ")
    if not os.environ["ASTRA_DB_TOKEN"].startswith("AstraCS:"):
        raise RuntimeError("ASTRA_DB_TOKEN no tiene el formato esperado (AstraCS:...)")

    # 3. Java 17 (PySpark 3.5 no soporta Java 21).
    if java_major() != 17:
        sh("apt-get install -y -qq openjdk-17-jre-headless > /dev/null")
    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] = f"{JAVA_HOME}/bin:" + os.environ["PATH"]

    # 4. Repo y dependencias fijadas en requirements.txt.
    if not (CLONE_DIR / "src").is_dir():
        sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {CLONE_DIR}")
    pins = [
        line.strip()
        for line in (CLONE_DIR / "requirements.txt").read_text().splitlines()
        if line.split("==")[0].strip() in PIPELINE_PACKAGES
    ]
    sh("pip install -q " + " ".join(pins))

    os.environ["PIPELINE_PROFILE"] = "cloud"
    os.environ["PYTHONPATH"] = str(CLONE_DIR)
    os.chdir(CLONE_DIR / "notebooks")

    import pyspark

    print(f"Python {sys.version.split()[0]} · Java {java_major()} · PySpark {pyspark.__version__} · perfil cloud")
    print("token: definido · bundle:", BUNDLE.name)
    if sys.version_info >= (3, 12):
        print("Aviso: PySpark 3.5 soporta oficialmente hasta Python 3.11; Colab trae", sys.version.split()[0])


In [ ]:
# Raíz del repo en sys.path: en Docker ya está (PYTHONPATH=/workspace); en un venv el notebook corre desde notebooks/.
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "config").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd

from src import pipeline
from src.common.config import load_config
from src.common.logs import setup_logging
from src.common.run_log import read_runs
from src.common.spark_session import build_spark

pd.set_option("display.max_columns", None)
setup_logging();

## Parámetros

| Parámetro | Valores | Efecto |
|---|---|---|
| `PROFILE` | `None`, `"local"`, `"cloud"` | `None` usa `PIPELINE_PROFILE` o `local` (D17) |
| `CLEAN` | `True` / `False` | Borra Bronze, Silver, Gold, Quarantine y los checkpoints **juntos** antes de correr (D13). Landing nunca se toca |
| `STEPS` | `None` o lista | `None` = todas las etapas por defecto. Siempre corren en el orden del pipeline |
| `SKIP` | lista | Etapas a saltear, por ejemplo `["serving"]` |
| `WITH_SANDBOX` | `True` / `False` | Agrega el sandbox de fixtures de calidad y late data (D20), en un lake aparte |

Para demostrar idempotencia (O6): correr una vez con `CLEAN = True`, después otra con `CLEAN = False`
y comparar el inventario de la última sección.

In [ ]:
PROFILE = None
CLEAN = False
STEPS = None
SKIP = []
WITH_SANDBOX = False

In [ ]:
cfg = load_config(PROFILE)
selected = pipeline.select_steps(STEPS, SKIP, WITH_SANDBOX)

print(cfg)
print("serving   :", cfg.get("serving.provider"), "· keyspace", cfg.get("serving.keyspace"))
print("lake      :", cfg.get("datalake.root"))
print("etapas    :", " → ".join(selected))

## Limpieza y reinicio

Solo si `CLEAN = True`. Borra las zonas generadas y los checkpoints en un solo paso: si se borrara
solo el checkpoint, el file sink de Bronze saltearía los lotes ya registrados sin escribir nada (D13).
El registro de corridas (`_metadata/`) se conserva: es la evidencia de las corridas anteriores.

In [ ]:
if CLEAN:
    pipeline.clean(cfg)
else:
    print("CLEAN = False: se reprocesa sobre el lake existente")

## SparkSession

Una sola sesión para todas las etapas. Lleva el connector de Cassandra solo si se carga el serving.

In [ ]:
spark = build_spark(cfg, app_name="pipeline_notebook", serving=pipeline.needs_connector(selected))
print("Spark", spark.version, "· UI en http://localhost:4040")

resumen = []
inicio = pd.Timestamp.now(tz="UTC").floor("s")   # para mostrar solo las corridas de esta sesión

def etapa(name):
    """Corre la etapa si está seleccionada y guarda su estado en `resumen`."""
    if name not in selected:
        print(f"{name}: no seleccionada")
        return None
    r = pipeline.run_step(name, cfg, spark)
    resumen.append(r)
    return r["result"] if r["status"] == "ok" else None

## 1 · Bronze batch: maestros

Los 7 CSV de Landing con esquema explícito, `ingest_ts`, `source_file` y deduplicación por PK
(D3, D11, D12). Control `landing = bronze + quarantine` por fuente (O3).

In [ ]:
masters = etapa("bronze_masters")
if masters:
    display(pd.DataFrame(masters).T)

## 2 · Bronze streaming: eventos de uso

Structured Streaming sobre `usage_events_stream/*.jsonl`: esquema explícito, watermark de 60 días,
`dropDuplicatesWithinWatermark("event_id")`, checkpoint y file sink en `append` (D6, D13).
Trigger `availableNow` para que termine solo (D18). Si el lake ya estaba cargado, procesa 0 archivos nuevos.

In [ ]:
stream = etapa("bronze_stream")
if stream:
    display(pd.DataFrame([stream["stream"]], index=["stream"]))
    display(pd.DataFrame([stream["bronze"]], index=["bronze total"]))

## 3 · Calidad

Reglas sobre los eventos de Bronze (D7, D19). Las que **bloquean** mandan la fila a Quarantine;
las que **marcan** la conservan con `dq_flags`. Control `bronze[d] = válidos[d] + quarantine[d]`
para cada fecha (O4). Evidencia en `evidence/entrega2/quality/`.

In [ ]:
quality = etapa("quality")
if quality:
    display(pd.Series({k: quality[k] for k in ("bronze", "valid", "quarantine", "cost_anomalies")}, name="filas").to_frame())
    display(pd.DataFrame({"bloquea": pd.Series(quality["blocked_by_rule"]), "marca": pd.Series(quality["flagged_by_rule"])}).fillna("—"))

## 4 · Silver

`dim_org` y `dim_resource`, eventos conformados (casteo de `value`, compatibilidad v1/v2) y
enriquecidos, y la tabla de features diarias `usage_daily_features` (D21). Overwrite dinámico por fecha (D10).

In [ ]:
silver = etapa("silver")
if silver:
    display(pd.DataFrame.from_dict({k: silver[k] for k in ("usage_events", "usage_daily_features")}, orient="index"))
    print("dimensiones:", silver["dims"], "· fechas sin balance:", len(silver["unbalanced_dates"]))

## 5 · Gold

Mart FinOps `org_daily_usage_by_service`, Top-N de 14 días `org_service_cost_14d` con `as_of_date`
(D14, D15) y `cost_anomaly_mart` con MAD por `(org_id, service)` (D9). Evidencia en `evidence/entrega2/gold/`.

In [ ]:
gold = etapa("gold")
if gold:
    print("as_of_date:", gold["as_of_date"])
    display(pd.DataFrame({"filas": gold["rows"], "archivos": gold["files"]}))
    display(pd.Series(gold["anomalies"], name="anomalías").to_frame())

## 6 · Serving

Carga de Gold en Cassandra (perfil local) o AstraDB (perfil cloud) con el connector, sobre las tablas
query-first de `src/serving/cql/02_tables.cql` (D8): upsert por clave natural en el mart diario y
`TRUNCATE` + carga en el Top-N (D14). Verifica `filas en Cassandra = filas en Gold` y corre las
consultas P1 y P2 de `03_queries.cql`; la evidencia queda en `evidence/entrega2/serving/queries.md`.
Para correr sin Cassandra: `SKIP = ["serving"]`.

In [ ]:
serving = etapa("serving")
if serving:
    print("provider:", serving["provider"], "· keyspace:", serving["keyspace"])
    display(pd.DataFrame(serving["tables"]).T)
    display(pd.Series(serving["queries"], name="filas devueltas").to_frame())

## 7 · Sandbox de calidad y late data (opcional)

Solo con `WITH_SANDBOX = True`. Mismo código, otro lake (`data/sandbox/`): fixtures con `event_id`
nulo y duplicado, `value` no numérico y un evento tardío que descarta el watermark (D20).

In [ ]:
sandbox = etapa("sandbox")
if sandbox:
    display(pd.Series({
        "tardías descartadas por watermark": sandbox["stream_late"]["dropped_by_watermark"],
        "en Quarantine": sandbox["quality"]["quarantine"],
    }, name="sandbox").to_frame())

## Resumen y evidencia

Estado de cada etapa, inventario del lake (filas, archivos, tamaño y particiones por tabla) y las
últimas líneas del registro de corridas (D11). Es la tabla de conteos para comparar entre corridas (O6).

In [ ]:
display(pd.DataFrame([{k: v for k, v in r.items() if k != "result"} for r in resumen]))

In [ ]:
inventario = pd.DataFrame(pipeline.lake_inventory(cfg, spark))
display(inventario)

In [ ]:
runs = pd.DataFrame(read_runs(cfg))
runs = runs[pd.to_datetime(runs["started_at"]) >= inicio]
display(runs[["started_at", "job", "status", "duration_s", "rows_read", "rows_written", "rows_rejected", "partitions_written"]])

In [ ]:
spark.stop()